In [1]:
import numpy as np
import pandas as pd
from skforecast.utils import save_forecaster
from skforecast.utils import load_forecaster

In [2]:
import sys
import os
sys.path.append(os.pardir)

In [3]:
# Exogenous features helpers
from features import set_holidays, cal_features, cyclic_features

In [22]:
from datetime import datetime
from datetime import timedelta

In [5]:
forecaster_loaded = load_forecaster('../model/forecaster_001.joblib', verbose=True, suppress_warnings=False)

/Users/maxwellgriffith/miniconda3/envs/loadcast/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ForecasterRecursive 
Estimator: LGBMRegressor 
Lags: [  1   2   3  20  21  22  23  24  25  26  27  28  29 146 167 170] 
Window features: None 
Window size: 170 
Series name: Demand 
Exogenous included: True 
Exogenous names: 
    Temperature, Holiday, week, day_of_week, hour, week_sin, week_cos, hour_sin,
    hour_cos, Temp_3D_Mean, Temp_2D_Max, Temp_2D_Min, Temp_1D_Min 
Categorical features: auto 
Transformer for y: None 
Transformer for exog: None 
Weight function included: False 
Differentiation order: None 
Drop NaN from series: False 
Training range: [Timestamp('2023-01-01 00:00:00'), Timestamp('2024-12-31 23:00:00')] 
Training index type: DatetimeIndex 
Training index frequency: <Hour> 
Estimator parameters: 
    {'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 1.0,
    'importance_type': 'split', 'learning_rate': 0.16546988609195726,
    'max_depth': 3, 'min_child_samples': 20, 'min_child_weight': 0.001,
    'min_split_gain': 0.0, 'n_estimators': 700, 'n_jobs'

- make regular predictions without retraining model using last_window
- This argument allows providing only the past values needed to create the autoregressive predictors ie lags
- When using the last_window argument, it is crucial to ensure that the length of last_window is sufficient to include the maximum lag (or custom predictor) used by the forecaster. For instance, if the forecaster employs lags 1, 24, and 48, last_window must include the most recent 48 values of the series

In [6]:
forecaster_loaded.last_window_

,Demand
date,
2024-12-24 22:00:00,30801.208
2024-12-24 23:00:00,31410.891
2024-12-25 00:00:00,31305.535
2024-12-25 01:00:00,30949.415
2024-12-25 02:00:00,30649.751
...,...
2024-12-31 19:00:00,32951.913
2024-12-31 20:00:00,32711.893
2024-12-31 21:00:00,32615.968


In [7]:
exo_vars = forecaster_loaded.exog_names_in_
exo_vars

['Temperature',
 'Holiday',
 'week',
 'day_of_week',
 'hour',
 'week_sin',
 'week_cos',
 'hour_sin',
 'hour_cos',
 'Temp_3D_Mean',
 'Temp_2D_Max',
 'Temp_2D_Min',
 'Temp_1D_Min']

In [8]:
#last window is the end of the validation split so we don't need to pull new load from gridstatus to tests the prod for now

In [9]:
df = (pd.read_csv("../data/raw/gsloadtemp_clean.csv")
        .drop_duplicates()
        .pipe(lambda df: df.set_index(pd.to_datetime(df["date"])))
        .drop(columns = ["date"])
     )
#this will throw an error if there are duplicates
df.index = df.index.tz_localize(None)
df.index.freq = 'h'

In [10]:
df.head()

,Demand,Temperature
date,,
2023-01-01 00:00:00,28771.933,6.90
2023-01-01 01:00:00,28488.282,7.70
2023-01-01 02:00:00,28073.965,6.55
2023-01-01 03:00:00,27756.863,5.80
2023-01-01 04:00:00,27271.343,5.65


In [11]:
TEST_START  = "2025-01-01 00:00:00"
TEST_END    = "2025-12-31 23:00:00"

In [12]:
data_test  = df.loc[TEST_START:, :].copy()

In [13]:
print(f"Test dates       : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})")

Test dates       : 2025-01-01 00:00:00 --- 2025-12-31 23:00:00  (n=8760)


- years worth of data to test on
- lags 170
- exogenous variables
    - Temperature
    - Holiday
    - week
    - day_of_week
    - hour
    - week_sin
    - week_cos
    - hour_sin
    - hour_cos
    - Temp_3D_Mean
    - Temp_2D_Max
    - Temp_2D_Min
    - Temp_1D_Min 

In [14]:
data_test = (data_test
            .pipe(set_holidays)
            .pipe(cal_features)
            .pipe(cyclic_features)
            .assign(Holiday=lambda d: d["Holiday"].astype(int))
            .assign(Temp_3D_Mean=lambda d: d["Temperature"].rolling("3D", center = False).mean())
            .assign(Temp_2D_Max =lambda d: d["Temperature"].rolling("2D", center = False).max())
            .assign(Temp_2D_Min =lambda d: d["Temperature"].rolling("2D", center = False).min())
            .assign(Temp_1D_Min =lambda d: d["Temperature"].rolling("1D", center = False).min())
           )

In [15]:
data_test.columns

Index(['Demand', 'Temperature', 'Holiday', 'month', 'week', 'day_of_week',
       'hour', 'month_sin', 'month_cos', 'week_sin', 'week_cos', 'day_sin',
       'day_cos', 'hour_sin', 'hour_cos', 'Temp_3D_Mean', 'Temp_2D_Max',
       'Temp_2D_Min', 'Temp_1D_Min'],
      dtype='str')

In [ ]:
data_test[exo_vars]

In [ ]:
forecaster_loaded.predict(
    steps = 24,
    exog = data_test[exo_vars]
)

In [18]:
#now we use "last window"
# need 170 a little over 7 days 
forecaster_loaded.window_size

170

In [19]:
#write function that takes the date you want to start predictions at and gives you the window before it

td170h = pd.Timedelta(170, "hours")
td170h

Timedelta('7 days 02:00:00')

In [23]:
datetime(2025, 2, 1, 0)

datetime.datetime(2025, 2, 1, 0, 0)

In [ ]:
datetime(2025, 2, 1, 0) - timedelta(hours=24)

In [ ]:
datetime(2025, 2, 1, 0).isoformat(' ')

In [ ]:
def get_last_window(window_size_hrs: int, target_date: datetime):
    window_start = target_date - timedelta(hours = window_size_hrs)
    return window_start

In [ ]:
get_last_window()

- Lets try and predict the first day in march 2024 so "2025-03-01 00" to "2025-03-01 23"

In [27]:
window_start = datetime(2025, 3, 1, 0)-timedelta(days = 8)
window_start

datetime.datetime(2025, 2, 21, 0, 0)

In [29]:
last_window_start = window_start.isoformat(' ')
last_window_start

'2025-02-21 00:00:00'

In [30]:
window_end = datetime(2025, 3, 1, 0)-timedelta(hours = 1)
window_end

datetime.datetime(2025, 2, 28, 23, 0)

In [32]:
last_window_end = window_end.isoformat(' ')
last_window_end

'2025-02-28 23:00:00'

In [ ]:
lw_data = data_test.loc[last_window_start:last_window_end, ["Demand"]].copy()
lw_data

In [ ]:
exo_predict = data_test.loc["2025-03-01 00:00:00": "2025-03-01 23:00:00", exo_vars].copy()
exo_predict

In [45]:
forecaster_loaded.predict(
    steps = 24,
    last_window = lw_data,
    exog = exo_predict
)

2025-03-01 00:00:00    30707.647054
2025-03-01 01:00:00    30761.629443
2025-03-01 02:00:00    30513.052892
2025-03-01 03:00:00    29978.072508
2025-03-01 04:00:00    29152.805919
2025-03-01 05:00:00    28460.056093
2025-03-01 06:00:00    28013.812587
2025-03-01 07:00:00    27658.142900
2025-03-01 08:00:00    27503.272058
2025-03-01 09:00:00    27631.532379
2025-03-01 10:00:00    28015.621057
2025-03-01 11:00:00    28698.498244
2025-03-01 12:00:00    29855.810693
2025-03-01 13:00:00    30768.329847
2025-03-01 14:00:00    30539.702748
2025-03-01 15:00:00    30154.346274
2025-03-01 16:00:00    29753.142780
2025-03-01 17:00:00    29264.604919
2025-03-01 18:00:00    28946.084007
2025-03-01 19:00:00    28786.231073
2025-03-01 20:00:00    28636.221921
2025-03-01 21:00:00    28665.735541
2025-03-01 22:00:00    28811.539055
2025-03-01 23:00:00    29300.174203
Freq: h, Name: pred, dtype: float64

- Now we need write functions that get all the correct dates for the previous window and the exo variables
- remeber that the most recent load actuals are D-1
- Predicting D+1
    - exogenous variables for D1 and D+1
    - load forecat for D-1

In [53]:
today_mn = datetime.today().replace(hour = 0, minute = 0, second = 0, microsecond = 0)
today_mn

datetime.datetime(2026, 8, 1, 0, 0)

In [56]:
today_mn + timedelta(hours = 47)

datetime.datetime(2026, 8, 2, 23, 0)

In [63]:
def get_date_ranges(day_one = None):
    dates = {}
    if day_one is None:
        day_one = datetime.today().replace(hour = 0, minute = 0, second = 0, microsecond = 0)
    
    dates["today"] = day_one
    #D-1 midnight to D-1 11pm
    lw_start = day_one-timedelta(days = 8)
    dates["lw_start"] = lw_start
    lw_end = day_one - timedelta(hours = 1)
    dates["lw_end"] = lw_end
    
    dates["exo_start"] = day_one
    exo_end = day_one + timedelta(hours = 47)
    dates["exo_end"] = exo_end

    return dates

In [64]:
pred_dates = get_date_ranges()
pred_dates

{'today': datetime.datetime(2026, 8, 1, 0, 0),
 'lw_start': datetime.datetime(2026, 7, 24, 0, 0),
 'lw_end': datetime.datetime(2026, 7, 31, 23, 0),
 'exo_start': datetime.datetime(2026, 8, 1, 0, 0),
 'exo_end': datetime.datetime(2026, 8, 2, 23, 0)}

In [62]:
{'today': datetime.datetime(2026, 8, 1, 0, 0),
 'lw_start': datetime.datetime(2026, 7, 24, 0, 0),
 'lw_end': datetime.datetime(2026, 7, 31, 23, 0),
 'exo_start': datetime.datetime(2026, 8, 1, 0, 0),
 'exo_end': datetime.datetime(2026, 8, 2, 23, 0)}

nothing here


In [65]:
test_date = datetime(2026, 9, 1, 0, 0)
test_date

datetime.datetime(2026, 9, 1, 0, 0)

In [67]:
test_date_ranges = get_date_ranges(test_date)
test_date_ranges

{'today': datetime.datetime(2026, 9, 1, 0, 0),
 'lw_start': datetime.datetime(2026, 8, 24, 0, 0),
 'lw_end': datetime.datetime(2026, 8, 31, 23, 0),
 'exo_start': datetime.datetime(2026, 9, 1, 0, 0),
 'exo_end': datetime.datetime(2026, 9, 2, 23, 0)}

- should probably add a check on day_one to make sure it's at midnight